In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import abc, gc, os, random, re, sys, time, traceback
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import stable_baselines3 as sb
from stable_baselines3.common.env_util import make_vec_env
from typing import SupportsFloat, Any
import gymnasium as gym
from gymnasium.core import ActType, ObsType

import faiss
import sklearn
from tqdm import tqdm
import chime
import scipy.sparse as sp

import torch
from torch.nn import functional as F
from torch.utils import data
from torch import optim

In [ ]:
from data_loaders import KuaiRecData, commit_dataset, CommitedDataset, JaggedCommittedDataset, commit_dataset_jagged
from dl_trainer import Trainer, sequence_collate, create_sequence_collator
from rs_algorithm.models import AdaptiveRepresentation
from rs_algorithm.models import TwoTowerRecommender, ThreeTowerRecommenderV2, ConcatInteractionHistoryRepresentationWithUserEmbedInitialization
from rs_algorithm.models.user_models.user_02_sim import DynamicPreferenceUserModel, norm_dot_product_based_relavance_calculation, SimUserModel

In [ ]:
from miscellaneous import plot_data

In [ ]:
import importlib

importlib.reload(sys.modules['rs_algorithm'])

In [ ]:
from rs_algorithm import IdealSolutionBaseline, Evaluator, SimulationEvaluator
from rs_algorithm import AlternatingLeastSquaresBaseline, BayesianPersonalizedRankingBaseline
from rs_algorithm import EmbedDotSolution, HistoryModelS1Solution, RLModelS2Solution

In [ ]:
data_model = KuaiRecData(root='../data/KuaiRec2.0', initial_specimen='smaller')
interactions = pd.read_parquet('../data/KuaiRec2.0/data/smaller_ratings_processed_sim_user.parquet')
data_model.ratings = interactions

In [ ]:
evaluator = Evaluator()
sim_evaluator = SimulationEvaluator(data_model, n_sample_users=10, n_time_stamps=500)

In [ ]:
%%time
ideal_solution = IdealSolutionBaseline(data_model)
als_solution = AlternatingLeastSquaresBaseline(data_model)
bpr_solution = BayesianPersonalizedRankingBaseline(data_model)
ed_solution = EmbedDotSolution(data_model)
s1_solution = HistoryModelS1Solution(data_model)
s2_solution = RLModelS2Solution(data_model)

In [ ]:
%%time
ideal_evaluation = evaluator.evaluate(ideal_solution)
als_evaluation = evaluator.evaluate(als_solution)
bpr_evaluation = evaluator.evaluate(bpr_solution)
ed_evaluation = evaluator.evaluate(ed_solution)
s1_evaluation = evaluator.evaluate(s1_solution)
s2_evaluation = evaluator.evaluate(s2_solution)

In [ ]:
%%time
ideal_results = sim_evaluator.evaluate_simulation(ideal_solution)
als_results = sim_evaluator.evaluate_simulation(als_solution)
bpr_results = sim_evaluator.evaluate_simulation(bpr_solution)
ed_results = sim_evaluator.evaluate_simulation(ed_solution)
s1_results = sim_evaluator.evaluate_simulation(s1_solution)
s2_results = sim_evaluator.evaluate_simulation(s2_solution)

In [ ]:
evaluations = [ideal_evaluation, als_evaluation, bpr_evaluation, ed_evaluation, s1_evaluation, s2_evaluation]


def plot_evaluations(evaluations, legend=None, sup_title='', kind='train', top_k='K'):
    fig = plt.figure(figsize=(14, 8.5))
    gs = GridSpec(3, 5)

    metric_bar_args = {'ylim': (-0.1, 1.1), 'legend': legend,
                       'dat_fn': lambda ax, dat: [ax.bar(i, v) for i, v in dat]}

    plot_data([(i, e[kind]['hit_rate_at_k']) for i, e in enumerate(evaluations)],
              title=f"Hit Rate @ {top_k}", ax=fig.add_subplot(gs[0, 0]), **metric_bar_args)
    plot_data([(i, e[kind]['precision_at_k']) for i, e in enumerate(evaluations)],
              title=f"Precision @ {top_k}", ax=fig.add_subplot(gs[0, 1]), **metric_bar_args)
    plot_data([(i, e[kind]['recall_at_k']) for i, e in enumerate(evaluations)],
              title=f"Recall @ {top_k}", ax=fig.add_subplot(gs[0, 2]), **metric_bar_args)
    plot_data([(i, e[kind]['ndcg_at_k']) for i, e in enumerate(evaluations)],
              title=f"NDCG @ {top_k}", ax=fig.add_subplot(gs[0, 3]), **metric_bar_args)
    plot_data([(i, e[kind]['mrr_at_k']) for i, e in enumerate(evaluations)],
              title=f"MRR @ {top_k}", ax=fig.add_subplot(gs[0, 4]), **metric_bar_args)

    plt.suptitle(sup_title)
    plt.tight_layout(), plt.show()


plot_evaluations(evaluations, ['ideal', 'als', 'bpr', 'ed', 's1', 's2'], 'T0 Recommendations')

In [ ]:
plot_evaluations(evaluations, ['ideal', 'als', 'bpr', 'ed', 's1', 's2'], 'T0 Recommendations', 'test')

In [ ]:
results = [ideal_results, als_results, bpr_results, ed_results, s1_results, s2_results]


def time_collapse_identity(arr): return arr


def time_collapse_mean(arr): return np.mean(arr, axis=0)


def time_collapse_last(arr): return arr[-1]


def aggregate_results(results, time_collapse=time_collapse_mean):
    # assert len(results) == n_solutions
    # assert len(results[i]) == n_sample_users
    # assert all metric keys exist in results[i][j]
    # assert for all core metrics M, results[i][j][M] is actually array
    # assert len(results[i][j][M]) == n_time_stamps
    # assert for all other metrics O, results[i][j][O] is actually a float
    #
    # This function averages each metric across users for every solution result.

    core_metrics = {'hit_rate_at_k', 'precision_at_k', 'recall_at_k', 'ndcg_at_k', 'mrr_at_k'}

    aggregate = [dict() for _ in results]
    for i, result in enumerate(results):
        for core_metric in core_metrics:
            series = [time_collapse(r[core_metric]) for r in result]
            if isinstance(series[0], list):
                aggregate[i][core_metric] = np.mean(series, axis=0)
            else:
                aggregate[i][core_metric] = np.mean(series)
        for other_metric in result[0].keys() - core_metrics:
            aggregate[i][other_metric] = np.mean([r[other_metric] for r in result])

    return aggregate


aggregate = aggregate_results(results, time_collapse_last)

In [ ]:
def plot_simulation_aggregates(aggregates, legend=None, sup_title='', kind='train', top_k='K'):
    fig = plt.figure(figsize=(14*4/5, 8.5))
    gs = GridSpec(3, 4)

    bar_args = {'dat_fn': lambda ax, dat: [ax.bar(i, v) for i, v in dat], 'legend': legend}
    metric_bar_args = {'ylim': (-0.1, 1.1), **bar_args}

    plot_data([(i, e['hit_rate_at_k']) for i, e in enumerate(aggregates)],
              title=f"Hit Rate @ {top_k}", ax=fig.add_subplot(gs[0, 0]), **metric_bar_args)
    plot_data([(i, e['precision_at_k']) for i, e in enumerate(aggregates)],
              title=f"Precision @ {top_k}", ax=fig.add_subplot(gs[0, 1]), **metric_bar_args)
    plot_data([(i, e['recall_at_k']) for i, e in enumerate(aggregates)],
              title=f"Recall @ {top_k}", ax=fig.add_subplot(gs[0, 2]), **metric_bar_args)
    plot_data([(i, e['ndcg_at_k']) for i, e in enumerate(aggregates)],
              title=f"NDCG @ {top_k}", ax=fig.add_subplot(gs[0, 3]), **metric_bar_args)
    # plot_data([(i, e['mrr_at_k']) for i, e in enumerate(aggregates)],
    #           title=f"MRR @ {top_k}", ax=fig.add_subplot(gs[0, 4]), **metric_bar_args)

    plot_data([(i, e['return_rate']) for i, e in enumerate(aggregates)],
              title='Return Rate', ax=fig.add_subplot(gs[1, 0]), **bar_args)
    plot_data([(i, e['acceptance_rate']) for i, e in enumerate(aggregates)],
              title='Acceptance Rate', ax=fig.add_subplot(gs[1, 1]), **bar_args)
    plot_data([(i, e['time_to_boredom']) for i, e in enumerate(aggregates)],
              title='Time to Boredom', ax=fig.add_subplot(gs[1, 2]), **bar_args)
    plot_data([(i, e['fatigue_score']) for i, e in enumerate(aggregates)],
              title='Fatigue Score', ax=fig.add_subplot(gs[1, 3]), **bar_args)

    plot_data([(i, e['content_repetition_rate']) for i, e in enumerate(aggregates)],
              title='content_repetition_rate', ax=fig.add_subplot(gs[2, 0]), **bar_args)
    plot_data([(i, e['novelty']) for i, e in enumerate(aggregates)],
              title='novelty', ax=fig.add_subplot(gs[2, 1]), **bar_args)
    plot_data([(i, e['intra_list_diversity']) for i, e in enumerate(aggregates)],
              title='intra_list_diversity', ax=fig.add_subplot(gs[2, 2]), **bar_args)
    plot_data([(i, e['time_to_discontent']) for i, e in enumerate(aggregates)],
              title='Time to Discontent', ax=fig.add_subplot(gs[2, 3]), **bar_args)
    # plot_data([(i, e['session_depth']) for i, e in enumerate(aggregates)],
    #           title='Session Depth', ax=fig.add_subplot(gs[1, 3]), **bar_args)

    plt.suptitle(sup_title)
    plt.tight_layout(), plt.show()


plot_simulation_aggregates(aggregate, sup_title='T500 Recommendations')

In [ ]:
aggregate = aggregate_results(results, lambda arr: arr[100])
plot_simulation_aggregates(aggregate, sup_title='T100 Recommendations')

In [ ]:
aggregate = aggregate_results(results, lambda arr: arr[300])
plot_simulation_aggregates(aggregate, sup_title='T300 Recommendations')

In [ ]:
def plot_simulation_results(results, legend=None, sup_title='', kind='train', top_k='K'):
    fig = plt.figure(figsize=(14, 8.5))
    gs = GridSpec(3, 5)

    plot_args = {'dat_fn': lambda ax, dat: [ax.plot(v) for v in dat], 'legend': legend}
    metric_plot_args = {'ylim': (-0.1, 1.1), **plot_args}
    aggregates = aggregate_results(results, time_collapse_identity)

    plot_data([e['hit_rate_at_k'] for e in aggregates],
              title=f"Hit Rate @ {top_k}", ax=fig.add_subplot(gs[0, 0]), **metric_plot_args)
    plot_data([e['precision_at_k'] for e in aggregates],
              title=f"Precision @ {top_k}", ax=fig.add_subplot(gs[0, 1]), **metric_plot_args)
    plot_data([e['recall_at_k'] for e in aggregates],
              title=f"Recall @ {top_k}", ax=fig.add_subplot(gs[0, 2]), **metric_plot_args)
    plot_data([e['ndcg_at_k'] for e in aggregates],
              title=f"NDCG @ {top_k}", ax=fig.add_subplot(gs[0, 3]), **metric_plot_args)
    plot_data([e['mrr_at_k'] for e in aggregates],
              title=f"MRR @ {top_k}", ax=fig.add_subplot(gs[0, 4]), **metric_plot_args)

    plt.suptitle(sup_title)
    plt.tight_layout(), plt.show()

plot_simulation_results(results)